# Matchbox FHIR → OMOP Demo
Covers server health checks, Echidna terminology lookups, and FHIR resource transforms.

In [1]:
import os
import json
import requests
import pandas as pd
from IPython.display import display

BASE_URL = os.environ.get('MATCHBOX_URL', 'http://matchbox:8080') + '/matchboxv3/fhir'
SCRIPTS_DIR = '/home/jovyan/matchbox_scripts'
ECHIDNA_URL = 'https://echidna.fhir.org/r4'

HEADERS_JSON = {'Accept': 'application/fhir+json'}
HEADERS_POST = {'Content-Type': 'application/fhir+json', 'Accept': 'application/fhir+json'}

print(f'Matchbox: {BASE_URL}')

Matchbox: http://matchbox:8080/matchboxv3/fhir


## 1. Server Health Checks

In [ ]:
# Verify OMOP IG $transform maps are loaded
r = requests.get(f'{BASE_URL}/StructureMap', headers=HEADERS_JSON)
bundle = r.json()

omop_maps = [
    {'name': e['resource'].get('name'), 'url': e['resource'].get('url')}
    for e in bundle.get('entry', [])
    if (e['resource'].get('url') or '').startswith('http://hl7.org/fhir/uv/omop/')
]

if omop_maps:
    print(f"OMOP StructureMaps loaded ({len(omop_maps)}) — $transform is ready:")
    display(pd.DataFrame(omop_maps))
else:
    print('WARNING: No OMOP StructureMaps found — IG may not be loaded')

In [3]:
# Check OMOP IG is loaded (check_ig_loaded.sh)
r = requests.get(f'{BASE_URL}/ImplementationGuide', params={'_content': 'omop'}, headers=HEADERS_JSON)
bundle = r.json()
total = bundle.get('total', 0)
print(f"OMOP IGs found: {total}")
for entry in bundle.get('entry', []):
    ig = entry['resource']
    print(f"  {ig.get('name')} {ig.get('version')}")

OMOP IGs found: 2
  None 1.0.0
  None 7.1.0


## 2. Terminology Lookups via Echidna

In [ ]:
def lookup_snomed(code, label=''):
    r = requests.get(
        f'{ECHIDNA_URL}/ConceptMap/$translate',
        headers=HEADERS_JSON,
        params={
            'system': 'http://snomed.info/sct',
            'code': code,
            'targetsystem': 'https://fhir-terminology.ohdsi.org',
        }
    )
    result = r.json()
    matched = next((p for p in result.get('parameter', []) if p['name'] == 'result'), {})
    concepts = [p for p in result.get('parameter', []) if p['name'] == 'match']
    print(f"[{label or code}] matched={matched.get('valueBoolean')}")
    for m in concepts:
        for part in m.get('part', []):
            if part['name'] == 'concept':
                c = part['valueCoding']
                print(f"  OMOP concept_id={c.get('code')}  display={c.get('display')}")
    return result

In [ ]:
# lookup_hypertension.sh
lookup_snomed('38341003', 'Hypertensive disorder');

In [ ]:
# lookup_fever.sh
lookup_snomed('386661006', 'Fever');

In [ ]:
# lookup_unknown.sh — bogus code, expect no match
lookup_snomed('0000001', 'Unknown/bogus code');

## 3. FHIR → OMOP Transforms

In [ ]:
OMOP_COLUMNS = {
    'ConditionOccurrence': [
        'condition_occurrence_id', 'person_id', 'condition_concept_id',
        'condition_start_date', 'condition_start_datetime', 'condition_end_date',
        'condition_end_datetime', 'condition_type_concept_id', 'condition_status_concept_id',
        'stop_reason', 'provider_id', 'visit_occurrence_id', 'visit_detail_id',
        'condition_source_value', 'condition_source_concept_id', 'condition_status_source_value',
    ],
    'ProcedureOccurrence': [
        'procedure_occurrence_id', 'person_id', 'procedure_concept_id',
        'procedure_date', 'procedure_datetime', 'procedure_end_date', 'procedure_end_datetime',
        'procedure_type_concept_id', 'modifier_concept_id', 'quantity',
        'provider_id', 'visit_occurrence_id', 'visit_detail_id',
        'procedure_source_value', 'procedure_source_concept_id', 'modifier_source_value',
    ],
    'Person': [
        'person_id', 'gender_concept_id', 'year_of_birth', 'month_of_birth', 'day_of_birth',
        'birth_datetime', 'race_concept_id', 'ethnicity_concept_id', 'location_id',
        'provider_id', 'care_site_id', 'person_source_value', 'gender_source_value',
        'gender_source_concept_id', 'race_source_value', 'race_source_concept_id',
        'ethnicity_source_value', 'ethnicity_source_concept_id',
    ],
}

def transform(resource, map_url, label=''):
    r = requests.post(
        f'{BASE_URL}/StructureMap/$transform',
        params={'source': map_url},
        headers=HEADERS_POST,
        json=resource,
    )
    result = r.json()
    rtype = result.get('resourceType', 'unknown')
    print(f"[{label}] status={r.status_code}  resourceType={rtype}")
    cols = OMOP_COLUMNS.get(rtype)
    if cols:
        display(pd.DataFrame([{c: result.get(c, '') for c in cols}]))
    else:
        print(json.dumps(result, indent=2))
    return result

def load(filename):
    with open(f'{SCRIPTS_DIR}/{filename}') as f:
        return json.load(f)

In [ ]:
# patient.sh — Patient → Person
transform(load('patient.json'), 'http://hl7.org/fhir/uv/omop/StructureMap/PersonMap', 'Patient→Person');

In [ ]:
# transform_condition.sh — Hypertension → ConditionOccurrence
transform(load('condition_hypertension.json'), 'http://hl7.org/fhir/uv/omop/StructureMap/ConditionMap', 'Hypertension');

In [ ]:
# transform_condition_fever.sh
transform(load('condition_fever.json'), 'http://hl7.org/fhir/uv/omop/StructureMap/ConditionMap', 'Fever');

In [ ]:
# transform_condition_refuted.sh — verificationStatus=refuted, expect suppressed/empty output
transform(load('condition_refuted.json'), 'http://hl7.org/fhir/uv/omop/StructureMap/ConditionMap', 'Refuted condition');

In [ ]:
# transform_condition_unknown_code.sh — bogus SNOMED code
transform(load('condition_unknown_code.json'), 'http://hl7.org/fhir/uv/omop/StructureMap/ConditionMap', 'Unknown SNOMED code');

In [ ]:
# transform_procedure_completed.sh
transform(load('procedure_completed.json'), 'http://hl7.org/fhir/uv/omop/StructureMap/ProcedureMap', 'Procedure completed');

In [ ]:
# transform_procedure_not_done.sh — status=not-done, expect suppressed output
transform(load('procedure_not_done.json'), 'http://hl7.org/fhir/uv/omop/StructureMap/ProcedureMap', 'Procedure not-done');

## 4. Upload PersonMap (FML)

In [ ]:
# upload_personmap.sh
with open(f'{SCRIPTS_DIR}/PersonMap.fml') as f:
    fml = f.read()

r = requests.post(
    f'{BASE_URL}/StructureMap',
    headers={'Content-Type': 'text/fhir-mapping', 'Accept': 'application/fhir+json'},
    data=fml.encode(),
)
print(f"Status: {r.status_code}")
result = r.json()
print(f"id={result.get('id')}  url={result.get('url')}")